# 🚀 CATI Phase 2 — End-to-End Fine-tuning

Jointly trains YOLOv11s backbone + CATI FiLM conditioning using detection loss.

**Prerequisites:**
- Phase 1 complete (`cati_best.pt` on Drive)
- YOLO dataset prepared (`yolo_dataset/` on Drive)
- T4 GPU runtime


In [ ]:
# Cell 1: Mount Drive + setup
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_DIR = '/content/sg-smart-city-analytics'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Suhxs-Reddy/sg-smart-city-analytics.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
    for k in list(sys.modules.keys()):
        if k.startswith('src.'): del sys.modules[k]

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

!pip install -q ultralytics torch torchvision pyyaml pillow

import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory/2**30:.1f} GB VRAM)')

FEATURE_DIR  = '/content/drive/MyDrive/sg_smart_city/data/features'
YOLO_DIR     = '/content/drive/MyDrive/sg_smart_city/data/yolo_dataset'
MODEL_DIR    = '/content/drive/MyDrive/sg_smart_city/models'
PHASE1_CKPT  = f'{MODEL_DIR}/cati_best.pt'
PHASE2_DIR   = f'{MODEL_DIR}/phase2'


In [ ]:
# Cell 2: Verify prerequisites
from pathlib import Path

checks = {
    'Phase 1 checkpoint': Path(PHASE1_CKPT).exists(),
    'YOLO dataset':        Path(YOLO_DIR).exists(),
    'data.yaml':           (Path(YOLO_DIR)/'data.yaml').exists(),
    'Train images':        len(list((Path(YOLO_DIR)/'images'/'train').glob('*'))) > 0,
    'Train labels':        len(list((Path(YOLO_DIR)/'labels'/'train').glob('*.txt'))) > 0,
}
for name, ok in checks.items():
    print(f'  {"✅" if ok else "❌"} {name}')

if not all(checks.values()):
    raise RuntimeError('Prerequisites not met — check above')
print('\n✅ All prerequisites met. Ready for Phase 2.')


In [ ]:
# Cell 3: Phase 2 Training
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', force=True)

from src.training.train_phase2 import CATIPhase2Trainer

trainer = CATIPhase2Trainer(
    yolo_dataset_dir=YOLO_DIR,
    feature_dir=FEATURE_DIR,
    cati_weights_path=PHASE1_CKPT,
    model_variant='yolo11s',
    epochs=20,
    batch_size=8,       # Reduce to 4 if OOM
    lr=1e-4,            # Low LR — backbone already pretrained
    device='cuda',
    freeze_backbone_epochs=3,
    save_dir=PHASE2_DIR,
)

results = trainer.train()
print('\n✅ Phase 2 complete!')
print(f'Checkpoints saved to: {PHASE2_DIR}')


In [ ]:
# Cell 4: Evaluate Phase 2 — CATI vs baseline mAP comparison
from ultralytics import YOLO
from pathlib import Path

data_yaml = str(Path(YOLO_DIR) / 'data.yaml')

# Baseline: standard YOLOv11s without CATI
print('Evaluating baseline YOLOv11s...')
baseline = YOLO('yolo11s.pt')
baseline_metrics = baseline.val(data=data_yaml, imgsz=640, device='cuda', verbose=False)
print(f'Baseline mAP50:   {baseline_metrics.box.map50:.4f}')
print(f'Baseline mAP50-95: {baseline_metrics.box.map:.4f}')

# CATI Phase 2: best YOLO weights from Phase 2 run
phase2_best = next(Path(PHASE2_DIR).glob('**/best.pt'), None)
if phase2_best:
    print(f'\nEvaluating CATI Phase 2 ({phase2_best.name})...')
    cati_model = YOLO(str(phase2_best))
    cati_metrics = cati_model.val(data=data_yaml, imgsz=640, device='cuda', verbose=False)
    print(f'CATI mAP50:        {cati_metrics.box.map50:.4f}')
    print(f'CATI mAP50-95:     {cati_metrics.box.map:.4f}')
    delta = cati_metrics.box.map50 - baseline_metrics.box.map50
    print(f'\nΔmAP50 (CATI - baseline): {delta:+.4f}')
else:
    print('No Phase 2 best.pt found — run Cell 3 first')
